# Global CO2 and Climate Change 1750-2023
### 50,000+ Records | 200+ Countries | EDA + Machine Learning
**Author:** Hassan Ali | [Kaggle: hassanali789](https://www.kaggle.com/hassanali789)

This notebook analyzes 270+ years of global CO2 and greenhouse gas emissions data covering 200+ countries. We explore historical trends, top emitters, per capita emissions, regional comparisons, and build an emissions prediction model.

**Source:** Our World in Data — Global Carbon Project

---


## 1. Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("husl")

print("All libraries loaded successfully!")

## 2. Load and Inspect Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/global-co2-climate-change-1750-2023/owid-co2-data.csv")

print(f"Shape        : {df.shape}")
print(f"Countries    : {df['country'].nunique()}")
print(f"Year range   : {df['year'].min()} to {df['year'].max()}")
print(f"Columns      : {list(df.columns)}")
print(f"\nMissing values (top 10):")
print(df.isnull().sum().sort_values(ascending=False).head(10))
df.head(10)

## 3. Statistical Summary

In [ ]:
key_cols = ["co2","co2_per_capita","cumulative_co2","methane","nitrous_oxide","temperature_change_from_co2"]
available = [c for c in key_cols if c in df.columns]

print("=== Key Metrics Statistics ===")
print(df[available].describe().round(3))

print("\n=== Records per Country (top 10) ===")
print(df['country'].value_counts().head(10))

## 4. Global CO2 Emissions Over Time

In [ ]:
world = df[df['country'] == 'World'].sort_values('year')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].fill_between(world['year'], world['co2'], alpha=0.3, color='#E53935')
axes[0].plot(world['year'], world['co2'], color='#E53935', linewidth=2)
axes[0].set_title('Global Annual CO2 Emissions (1750-2023)', fontsize=13, fontweight='bold', pad=12)
axes[0].set_ylabel('CO2 Emissions (Million Tonnes)')
axes[0].set_xlabel('Year')

axes[1].fill_between(world['year'], world['cumulative_co2'], alpha=0.3, color='#B71C1C')
axes[1].plot(world['year'], world['cumulative_co2'], color='#B71C1C', linewidth=2)
axes[1].set_title('Cumulative CO2 Emissions (1750-2023)', fontsize=13, fontweight='bold', pad=12)
axes[1].set_ylabel('Cumulative CO2 (Million Tonnes)')
axes[1].set_xlabel('Year')

plt.tight_layout()
plt.savefig('global_co2_trend.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"CO2 in 1900 : {world[world['year']==1900]['co2'].values[0]:.0f} Mt")
print(f"CO2 in 2023 : {world[world['year']==world['year'].max()]['co2'].values[0]:.0f} Mt")

## 5. Top 15 CO2 Emitting Countries (2022)

In [ ]:
latest_year = df[df['country'] != 'World']['year'].max()
latest = df[(df['year'] == latest_year) & (df['country'] != 'World')].dropna(subset=['co2'])
top15 = latest.nlargest(15, 'co2')

fig, ax = plt.subplots(figsize=(12, 6))
colors = plt.cm.YlOrRd(np.linspace(0.3, 1.0, 15))[::-1]
bars = ax.barh(top15['country'], top15['co2'],
               color=colors, edgecolor='white', linewidth=0.3)
ax.bar_label(bars, fmt='%.0f Mt', padding=3, fontsize=8)
ax.set_title(f'Top 15 CO2 Emitters ({latest_year})', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Annual CO2 Emissions (Million Tonnes)')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('top15_emitters.png', dpi=150, bbox_inches='tight')
plt.show()

print(top15[['country','co2','co2_per_capita']].to_string(index=False))

## 6. CO2 Per Capita — Fair Share Analysis

In [ ]:
latest_pc = df[(df['year'] == latest_year) & (df['country'] != 'World')].dropna(subset=['co2_per_capita'])
top15_pc = latest_pc.nlargest(15, 'co2_per_capita')
bot15_pc = latest_pc.nsmallest(15, 'co2_per_capita')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

bars = axes[0].barh(top15_pc['country'], top15_pc['co2_per_capita'],
                    color='#E53935', edgecolor='white', linewidth=0.3)
axes[0].bar_label(bars, fmt='%.1f t', padding=3, fontsize=8)
axes[0].set_title(f'Highest CO2 Per Capita ({latest_year})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('CO2 Per Capita (Tonnes/Person)')
axes[0].invert_yaxis()

bars = axes[1].barh(bot15_pc['country'], bot15_pc['co2_per_capita'],
                    color='#43A047', edgecolor='white', linewidth=0.3)
axes[1].bar_label(bars, fmt='%.2f t', padding=3, fontsize=8)
axes[1].set_title(f'Lowest CO2 Per Capita ({latest_year})', fontsize=12, fontweight='bold')
axes[1].set_xlabel('CO2 Per Capita (Tonnes/Person)')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('co2_per_capita.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Historical Emissions — Major Countries

In [ ]:
major_countries = ['United States', 'China', 'India', 'Russia', 'Germany',
                   'United Kingdom', 'Japan', 'Pakistan']
major_countries = [c for c in major_countries if c in df['country'].unique()]

fig, ax = plt.subplots(figsize=(13, 6))
for country in major_countries:
    c_df = df[(df['country'] == country) & (df['year'] >= 1900)].sort_values('year')
    ax.plot(c_df['year'], c_df['co2'], linewidth=2, label=country)

ax.set_title('Annual CO2 Emissions — Major Countries (1900-2023)', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('CO2 Emissions (Million Tonnes)')
ax.set_xlabel('Year')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('major_countries_trend.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Share of Global Emissions (2022)

In [ ]:
share = df[(df['year'] == latest_year) & (df['country'] != 'World')].dropna(subset=['share_global_co2'])
top10_share = share.nlargest(10, 'share_global_co2')
others_share = 100 - top10_share['share_global_co2'].sum()
pie_data = list(top10_share['share_global_co2']) + [others_share]
pie_labels = list(top10_share['country']) + ['Rest of World']

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(pie_data)))
wedges, texts, autotexts = ax.pie(pie_data, labels=pie_labels, autopct='%1.1f%%',
                                   colors=colors, startangle=140,
                                   pctdistance=0.8, textprops={'fontsize': 9})
ax.set_title(f'Share of Global CO2 Emissions ({latest_year})', fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('emission_share.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. CO2 Sources Breakdown

In [ ]:
world_recent = df[(df['country'] == 'World') & (df['year'] >= 1950)].sort_values('year')

source_cols = ['coal_co2', 'oil_co2', 'gas_co2', 'cement_co2', 'flaring_co2']
available_sources = [c for c in source_cols if c in world_recent.columns]

fig, ax = plt.subplots(figsize=(13, 6))
ax.stackplot(world_recent['year'],
             [world_recent[c].fillna(0) for c in available_sources],
             labels=[c.replace('_co2','').title() for c in available_sources],
             alpha=0.85)
ax.set_title('Global CO2 Emissions by Source (1950-2023)', fontsize=14, fontweight='bold', pad=12)
ax.set_ylabel('CO2 Emissions (Million Tonnes)')
ax.set_xlabel('Year')
ax.legend(loc='upper left', fontsize=9)
plt.tight_layout()
plt.savefig('co2_sources.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Temperature Change from CO2

In [ ]:
if 'temperature_change_from_co2' in df.columns:
    world_temp = df[(df['country'] == 'World') & (df['year'] >= 1850)].sort_values('year')

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.fill_between(world_temp['year'], world_temp['temperature_change_from_co2'],
                    where=world_temp['temperature_change_from_co2'] > 0,
                    alpha=0.4, color='#E53935', label='Warming')
    ax.plot(world_temp['year'], world_temp['temperature_change_from_co2'],
            color='#E53935', linewidth=2)
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.axhline(1.5, color='orange', linewidth=1.5, linestyle='--', label='Paris Agreement target (1.5C)')
    ax.axhline(2.0, color='red', linewidth=1.5, linestyle='--', label='Critical threshold (2.0C)')
    ax.set_title('Global Temperature Change from CO2 (1850-2023)', fontsize=14, fontweight='bold', pad=12)
    ax.set_ylabel('Temperature Change (C)')
    ax.set_xlabel('Year')
    ax.legend()
    plt.tight_layout()
    plt.savefig('temperature_change.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("temperature_change_from_co2 column not available in this dataset version")

## 11. Pakistan CO2 Profile

In [ ]:
pak = df[df['country'] == 'Pakistan'].sort_values('year')
world_avg = df[df['country'] == 'World'].sort_values('year')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(pak['year'], pak['co2'], color='#43A047', linewidth=2.5, label='Pakistan')
axes[0].set_title('Pakistan Annual CO2 Emissions', fontsize=13, fontweight='bold')
axes[0].set_ylabel('CO2 (Million Tonnes)')
axes[0].set_xlabel('Year')
axes[0].legend()

if 'co2_per_capita' in pak.columns:
    pak_pc = pak.dropna(subset=['co2_per_capita'])
    axes[1].plot(pak_pc['year'], pak_pc['co2_per_capita'],
                 color='#43A047', linewidth=2.5, label='Pakistan')
    axes[1].set_title('Pakistan CO2 Per Capita', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('CO2 Per Capita (Tonnes/Person)')
    axes[1].set_xlabel('Year')
    axes[1].legend()

plt.tight_layout()
plt.savefig('pakistan_co2.png', dpi=150, bbox_inches='tight')
plt.show()

latest_pak = pak[pak['year'] == pak['year'].max()]
print(f"Pakistan latest CO2       : {latest_pak['co2'].values[0]:.1f} Mt")
print(f"Pakistan CO2 per capita   : {latest_pak['co2_per_capita'].values[0]:.2f} t/person")
print(f"Pakistan share of global  : {latest_pak['share_global_co2'].values[0]:.2f}%")

## 12. Feature Engineering for Machine Learning

In [ ]:
df_ml = df[df['country'] != 'World'].copy()
df_ml = df_ml[df_ml['year'] >= 1950].sort_values(['country','year'])
df_ml = df_ml.dropna(subset=['co2','population'])

le = LabelEncoder()
df_ml['country_enc'] = le.fit_transform(df_ml['country'])

df_ml['co2_lag1']    = df_ml.groupby('country')['co2'].shift(1)
df_ml['co2_lag5']    = df_ml.groupby('country')['co2'].shift(5)
df_ml['co2_growth']  = df_ml.groupby('country')['co2'].pct_change()
df_ml['log_pop']     = np.log1p(df_ml['population'])
df_ml['year_norm']   = df_ml['year'] - 1950

if 'gdp' in df_ml.columns:
    df_ml['log_gdp'] = np.log1p(df_ml['gdp'].fillna(0))
else:
    df_ml['log_gdp'] = 0

df_ml = df_ml.dropna(subset=['co2_lag1','co2_lag5'])
df_ml = df_ml.replace([np.inf, -np.inf], np.nan).dropna(subset=['co2_lag1','co2_lag5','co2_growth'])

print(f"ML-ready rows : {len(df_ml):,}")
df_ml[['country','year','co2','co2_lag1','co2_growth','log_pop']].head(10)

## 13. Model Training — Predicting CO2 Emissions

In [ ]:
features = ['co2_lag1','co2_lag5','co2_growth','log_pop','log_gdp','year_norm','country_enc']

X = df_ml[features].fillna(0)
y = df_ml['co2']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]:,}")
print(f"Test size : {X_test.shape[0]:,}")
print()

models = {
    "Linear Regression" : LinearRegression(),
    "Random Forest"     : RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting" : GradientBoostingRegressor(n_estimators=100, random_state=42),
}

results = []
trained = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    mae  = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds) ** 0.5
    r2   = r2_score(y_test, preds)
    results.append({"Model": name, "MAE": round(mae,2), "RMSE": round(rmse,2), "R2": round(r2,4)})
    trained[name] = (model, preds)
    print(f"{name:22s} -> MAE: {mae:.2f} | RMSE: {rmse:.2f} | R2: {r2:.4f}")

## 14. Model Comparison and Feature Importance

In [ ]:
results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
best_name  = results_df.iloc[0]["Model"]
best_preds = trained[best_name][1]
best_model = trained[best_name][0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(results_df["Model"], results_df["R2"],
                   color=["#43A047","#1E88E5","#FB8C00"],
                   edgecolor="white", linewidth=0.3)
for i, (_, row) in enumerate(results_df.iterrows()):
    axes[0].text(i, row["R2"]+0.005, f"R2={row['R2']}", ha="center", fontsize=9)
axes[0].set_title("Model R2 Score Comparison", fontsize=13, fontweight="bold")
axes[0].set_ylabel("R2 Score")
axes[0].set_ylim(0, 1.1)
axes[0].tick_params(axis="x", rotation=10)

if hasattr(best_model, "feature_importances_"):
    importances = pd.Series(best_model.feature_importances_, index=features).sort_values()
    axes[1].barh(importances.index, importances.values, color="#66BB6A")
    axes[1].set_title(f"Feature Importance — {best_name}", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Importance Score")

plt.tight_layout()
plt.savefig("model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Best Model: {best_name}")
print(results_df.to_string(index=False))

## 15. Actual vs Predicted

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, best_preds, alpha=0.2, s=8, color="#E53935")
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
axes[0].plot(lims, lims, "b--", linewidth=1.2, label="Perfect prediction")
axes[0].set_xlabel("Actual CO2 Emissions (Mt)")
axes[0].set_ylabel("Predicted CO2 Emissions (Mt)")
axes[0].set_title(f"Actual vs Predicted — {best_name}", fontsize=13, fontweight="bold")
axes[0].legend()

residuals = y_test.values - best_preds
axes[1].hist(residuals, bins=50, color="#42A5F5", edgecolor="white", linewidth=0.3)
axes[1].axvline(0, color="red", linestyle="--", linewidth=1.2)
axes[1].set_title("Residual Distribution", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Residual (Mt)")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig("actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

## 16. Key Findings and Conclusions

### Emissions Trends
- Global CO2 emissions have risen dramatically since the Industrial Revolution, especially post-1950
- China and the United States together account for over 40% of global annual emissions
- Per capita emissions tell a different story — small Gulf states and Australia lead on a per-person basis

### Climate Impact
- Global temperature has already risen close to 1.5C above pre-industrial levels
- The Paris Agreement target of limiting warming to 1.5C is becoming increasingly difficult to achieve
- Coal remains the single largest source of CO2 emissions globally

### Pakistan
- Pakistan contributes less than 1% of global CO2 emissions despite being the 5th most populous country
- This highlights the climate injustice debate — low emitters face the worst climate impacts

### Machine Learning
- Previous year emissions (lag features) are the strongest predictors — emissions change gradually
- Random Forest and Gradient Boosting achieve very high R2 scores due to strong temporal autocorrelation
- The model can be used to forecast near-term emissions trajectories for policy planning

---

Dataset by Hassan Ali | hassanali789 on Kaggle
Source: Our World in Data — Global Carbon Project
